# Trabajo Práctico II - Funciones de Distancia, continuación

Gestión Avanzada de Datos - UTN FRCU

Alumno: Manuel Riedel

Objetivo: representar situaciones reales y resolver búsquedas por similitud con funciones adecuadas para conjuntos, vectores, histogramas, distribuciones y series temporales.

Indicaciones generales. En cada ejercicio, explicitar la representación, mostrar al menos un cálculo completo y justificar la elección de la función en el contexto. Redondear los resultados finales a cuatro decimales y ordenar de menor a mayor distancia. Las consultas de rango incluyen el límite. En k-NN, resolver empates por identificador alfabético. Junto con la resolución, entregar el código desarrollado para realizar los cálculos (en Python).

Los resultados están redondeados a cuatro decimales, sin redondear los pasos intermedios, y ordenados de menor a mayor distancia. Las consultas de rango incluyen el límite y los empates en k-NN se resuelven por identificador alfabético. Uso punto decimal y coma para separar coordenadas, igual que en la salida de Python.

In [1]:
import math

import pandas as pd

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_colwidth", None)


def f4(x):
    return f"{x:.4f}"


def ordenar(distancias):
    # de menor a mayor distancia, con los empates por identificador alfabético
    # redondeo a 10 decimales para que el error de punto flotante no rompa un empate
    return sorted(distancias.items(), key=lambda par: (round(par[1], 10), par[0]))


def k_vecinos(distancias, k):
    return [objeto for objeto, _ in ordenar(distancias)[:k]]


def rango(distancias, r):
    # el límite se incluye
    return [objeto for objeto, d in ordenar(distancias) if round(d, 10) <= r]


def tabla(distancias, nombre="Distancia"):
    return pd.DataFrame(ordenar(distancias), columns=["Objeto", nombre]).set_index("Objeto")

## Ejercicio 1. Jaccard y hábitos de compra

Un supermercado busca clientes con hábitos parecidos para sugerir productos. Solo interesa qué categorías compraron durante el mes. Las cantidades y las categorías que ninguno compró no deben influir.

| Cliente | Categorías compradas |
|---|---|
| Q | pan, leche, fruta, café |
| A | pan, leche, fruta, té |
| B | pan, leche |
| C | carne, arroz, aceite |
| D | pan, leche, fruta, café, yogur |

$$d_J(A, B) = 1 - \frac{|A \cap B|}{|A \cup B|}$$

a) Calcular las distancias de Q a A, B, C y D, indicando cada intersección y unión.

b) Obtener los 2 vecinos más cercanos de Q y la consulta de rango con radio 0.5.

c) ¿Cambian las distancias si se agregan al catálogo 100 categorías que nadie compró? ¿Y si Q compra diez veces más unidades de las mismas categorías?

d) Proponer una convención para comparar dos historiales vacíos y otra para comparar uno vacío con uno no vacío. Justificarla.

Cada cliente se representa como el conjunto de categorías que compró en el mes. Se pierden las cantidades y el orden de compra. Uso Jaccard porque al supermercado solo le interesa qué categorías compró cada uno, y Jaccard compara lo que tienen en común contra todo lo que compraron entre los dos, sin contar lo que ninguno compró.

In [2]:
CLIENTES = {"Q": {"pan", "leche", "fruta", "café"},
            "A": {"pan", "leche", "fruta", "té"},
            "B": {"pan", "leche"},
            "C": {"carne", "arroz", "aceite"},
            "D": {"pan", "leche", "fruta", "café", "yogur"}}


def jaccard(a, b):
    if not a and not b:
        return 0.0  # dos historiales vacíos, convención del inciso d
    return 1 - len(a & b) / len(a | b)


q = CLIENTES["Q"]
d1 = {c: jaccard(q, CLIENTES[c]) for c in "ABCD"}

pd.DataFrame([(c, ", ".join(sorted(q & CLIENTES[c])), ", ".join(sorted(q | CLIENTES[c])),
               len(q & CLIENTES[c]), len(q | CLIENTES[c]), d)
              for c, d in ordenar(d1)],
             columns=["Cliente", "Intersección con Q", "Unión con Q", "|∩|", "|∪|", "dJ"]).set_index("Cliente")

,Intersección con Q,Unión con Q,|∩|,|∪|,dJ
Cliente,,,,,
D,"café, fruta, leche, pan","café, fruta, leche, pan, yogur",4,5,0.2000
A,"fruta, leche, pan","café, fruta, leche, pan, té",3,5,0.4000
B,"leche, pan","café, fruta, leche, pan",2,4,0.5000
C,,"aceite, arroz, café, carne, fruta, leche, pan",0,7,1.0000


a) Cálculo completo para A: Q ∩ A = {pan, leche, fruta}, que tiene 3 elementos. Q ∪ A = {pan, leche, fruta, café, té}, que tiene 5. Entonces dJ(Q, A) = 1 − 3/5 = 0.4000.

In [3]:
print("2 vecinos más cercanos:", k_vecinos(d1, 2))
print("Rango con radio 0.5:", rango(d1, 0.5))

2 vecinos más cercanos: ['D', 'A']
Rango con radio 0.5: ['D', 'A', 'B']


b) Los 2 vecinos más cercanos de Q son D y A. La consulta de rango con radio 0.5 devuelve D, A y B. B está justo en el límite y el límite se incluye.

c) No cambian en ninguno de los dos casos. Las categorías que nadie compró no entran ni en la intersección ni en la unión. Y como el conjunto solo registra si una categoría se compró o no, comprar diez veces más de las mismas categorías deja a Q con el mismo conjunto.

In [4]:
print("Dos vacíos:", f4(jaccard(set(), set())))
print("Uno vacío y uno no vacío:", f4(jaccard(set(), q)))

Dos vacíos: 0.0000
Uno vacío y uno no vacío: 1.0000


d) Dos historiales vacíos dan 0/0, así que la fórmula no está definida. Propongo tomar d = 0, porque los dos conjuntos son iguales y así se mantiene que la distancia de un objeto consigo mismo es 0. De todos modos no los usaría para recomendar, porque un cliente sin compras no dice nada sobre sus hábitos. Uno vacío con uno no vacío da d = 1, que es lo que ya sale de la fórmula (intersección 0 y unión mayor que 0): no tienen nada en común.

## Ejercicio 2. Hamming y detección de imágenes duplicadas

Una galería representa cada imagen mediante un hash perceptual de 8 bits. Los bits ocupan posiciones fijas y todos tienen el mismo peso. Se busca detectar posibles duplicados de Q = 10110110.

A = 10110111, B = 10000110, C = 01001001, D = 10100110.

$d_H(x, y)$ = cantidad de posiciones en las que x e y difieren

a) Calcular las cuatro distancias e identificar los candidatos con distancia menor o igual que 1.

b) Si una imagen llega con un hash de 7 bits, ¿puede compararse directamente? Explicar qué debe uniformarse y por qué no corresponde alinear los bits como si fueran palabras.

c) ¿Un hash cercano garantiza que dos imágenes sean iguales? Explicar cómo usar el resultado como filtro de candidatos.

Cada imagen se representa con un hash de 8 bits, donde cada posición siempre significa lo mismo y todas pesan igual. Uso Hamming porque compara posición por posición y cuenta cuántos bits cambian, que es lo que importa cuando las posiciones son fijas y tienen el mismo peso.

In [5]:
Q_HASH = "10110110"
HASHES = {"A": "10110111", "B": "10000110", "C": "01001001", "D": "10100110"}


def hamming(x, y):
    if len(x) != len(y):
        raise ValueError("Hamming necesita cadenas del mismo largo")
    return sum(1 for a, b in zip(x, y) if a != b)


d2 = {i: hamming(Q_HASH, h) for i, h in HASHES.items()}

pd.DataFrame([(i, HASHES[i], [p + 1 for p in range(8) if Q_HASH[p] != HASHES[i][p]], d)
              for i, d in ordenar(d2)],
             columns=["Imagen", "Hash", "Posiciones distintas", "dH"]).set_index("Imagen")

,Hash,Posiciones distintas,dH
Imagen,,,
A,10110111,[8],1
D,10100110,[4],1
B,10000110,"[3, 4]",2
C,01001001,"[1, 2, 3, 4, 5, 6, 7, 8]",8


In [6]:
print("Candidatos con distancia menor o igual que 1:", rango(d2, 1))

Candidatos con distancia menor o igual que 1: ['A', 'D']


a) Cálculo completo para B: Q = 10110110 y B = 10000110 difieren en las posiciones 3 y 4, entonces dH(Q, B) = 2. Como la distancia es una cantidad de posiciones, sale entera. Los candidatos con distancia menor o igual que 1 son A y D. C es el complemento de Q, difiere en los 8 bits.

In [7]:
try:
    hamming(Q_HASH, "1011011")
except ValueError as error:
    print("Hash de 7 bits:", error)

Hash de 7 bits: Hamming necesita cadenas del mismo largo


b) No se puede comparar directo, porque Hamming necesita cadenas del mismo largo. Hay que uniformar el método: calcular el hash de esa imagen con el mismo algoritmo y los mismos 8 bits que el resto de la galería. No corresponde alinear los bits como si fueran palabras (insertar o correr bits, como en Levenshtein), porque cada posición tiene un significado fijo. Si se corre un bit, se compara una característica de la imagen con otra distinta.

c) No. El hash resume la imagen en 8 bits, así que hay solo 256 valores posibles e imágenes distintas pueden tener el mismo hash o uno muy parecido. Por eso lo uso como filtro: primero me quedo con las imágenes a distancia menor o igual que 1 (acá A y D), que es barato de calcular, y después a esos candidatos los verifico con una comparación más precisa, como comparar las imágenes completas o revisarlas a mano. B y C se descartan sin gastar esa comparación.

## Ejercicio 3. Mahalanobis y perfiles de funcionamiento

Una fábrica registra temperatura y presión de una máquina. Ambas suelen aumentar juntas. Se buscan registros históricos cercanos al funcionamiento de referencia q = (0, 0), expresado como desviaciones respecto de los valores habituales. Las variables ya fueron estandarizadas.

La covarianza histórica es $\Sigma = \begin{bmatrix} 1 & 0.8 \\ 0.8 & 1 \end{bmatrix}$. Utilizar su inversa $\Sigma^{-1} = \frac{1}{0.36} \begin{bmatrix} 1 & -0.8 \\ -0.8 & 1 \end{bmatrix}$, sin volver a estimarla con los candidatos.

| Registro | Temperatura estandarizada | Presión estandarizada |
|---|---|---|
| A | 1 | 1 |
| B | 1 | −1 |
| C | 0.5 | 0 |

$$d_M(x, q) = \sqrt{(x - q)^T \Sigma^{-1} (x - q)}$$

a) Calcular las distancias de Mahalanobis y recuperar los 2 registros más cercanos a q.

b) Calcular, como contraste, las distancias euclidianas. Comparar especialmente A y B e interpretar el efecto de la correlación.

c) Recuperar los registros con dM ≤ 1.5. Explicar por qué una desviación conjunta puede resultar más habitual que una variación en sentidos opuestos.

d) ¿Qué condición debe cumplir Σ para aplicar esta fórmula y obtener una métrica? ¿Qué problema aparece si dos variables son copias exactas?

Cada registro se representa como un vector (temperatura, presión) estandarizado. Uso Mahalanobis porque las dos variables están correlacionadas y esta distancia tiene en cuenta la covarianza: mide qué tan raro es un registro según cómo se comportan juntas las variables, no solo qué tan lejos está.

In [8]:
REGISTROS = {"A": (1, 1), "B": (1, -1), "C": (0.5, 0)}
Q_REF = (0, 0)
INVERSA = [[1 / 0.36, -0.8 / 0.36],
           [-0.8 / 0.36, 1 / 0.36]]


def mahalanobis(x, q, inversa):
    dif = [a - b for a, b in zip(x, q)]
    n = len(dif)
    producto = [sum(inversa[i][j] * dif[j] for j in range(n)) for i in range(n)]
    return math.sqrt(sum(dif[i] * producto[i] for i in range(n)))


def euclidiana(x, y):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(x, y)))


d3 = {r: mahalanobis(v, Q_REF, INVERSA) for r, v in REGISTROS.items()}
e3 = {r: euclidiana(v, Q_REF) for r, v in REGISTROS.items()}

pd.DataFrame([(r, REGISTROS[r], d, e3[r]) for r, d in ordenar(d3)],
             columns=["Registro", "Vector", "dM", "Euclidiana"]).set_index("Registro")

,Vector,dM,Euclidiana
Registro,,,
C,"(0.5, 0)",0.8333,0.5000
A,"(1, 1)",1.0541,1.4142
B,"(1, -1)",3.1623,1.4142


In [9]:
print("2 registros más cercanos:", k_vecinos(d3, 2))

2 registros más cercanos: ['C', 'A']


a) Cálculo completo para A: x − q = (1, 1). Multiplicando por la inversa, (1/0.36) x (1 − 0.8, −0.8 + 1) = (1/0.36) x (0.2, 0.2). Multiplicando por (x − q) queda (0.2 + 0.2) / 0.36 = 1.1111, y dM(A, q) = √1.1111 = 1.0541. Los 2 registros más cercanos a q son C y A.

b) Con la euclidiana A y B empatan en 1.4142. Con Mahalanobis B queda tres veces más lejos que A (3.1623 contra 1.0541). En A las dos variables suben juntas, que es lo habitual porque la correlación es 0.8. En B una sube y la otra baja, que casi nunca pasa. La euclidiana no ve esa diferencia porque trata a las variables como si no tuvieran relación.

In [10]:
print("Registros con dM <= 1.5:", rango(d3, 1.5))
print("A en la dirección (1, 1):", f4(2 / 1.8))
print("B en la dirección (1, -1):", f4(2 / 0.2))

Registros con dM <= 1.5: ['C', 'A']
A en la dirección (1, 1): 1.1111
B en la dirección (1, -1): 10.0000


c) Con dM menor o igual que 1.5 se recuperan C y A. Una desviación conjunta es más habitual porque, con correlación positiva, los datos históricos se reparten sobre una elipse alargada en la dirección (1, 1). En esa dirección la varianza es 1 + 0.8 = 1.8 y en la dirección (1, −1) es 1 − 0.8 = 0.2. Por eso moverse en sentidos opuestos cuesta mucho más, aunque el desplazamiento sea igual de largo: el cuadrado de la distancia de A es 2 / 1.8 = 1.1111 y el de B es 2 / 0.2 = 10.

In [11]:
# dos variables que son copias exactas tienen correlación 1
copias = [[1, 1], [1, 1]]
print("Determinante con copias exactas:", copias[0][0] * copias[1][1] - copias[0][1] * copias[1][0])
print("Determinante de la covarianza del ejercicio:", round(1 * 1 - 0.8 * 0.8, 4))

Determinante con copias exactas: 0
Determinante de la covarianza del ejercicio: 0.36


d) Σ tiene que ser simétrica y definida positiva, es decir, con todos sus autovalores mayores que cero. Así tiene inversa y la distancia es una métrica. Si dos variables son copias exactas, la correlación es 1, Σ = [[1, 1], [1, 1]] tiene determinante 0 y no tiene inversa, así que no se puede aplicar la fórmula. Habría que sacar una de las dos copias.

## Ejercicio 4. Chi cuadrado y búsqueda por textura

Un catálogo de telas usa histogramas de tres patrones locales de textura. Las categorías son diferentes tipos de patrón, sin una distancia espacial definida entre ellas. Cada imagen aporta 10 observaciones, obtenidas con el mismo procedimiento.

| Imagen | Patrón 1 | Patrón 2 | Patrón 3 |
|---|---|---|---|
| Q | 4 | 0 | 6 |
| A | 2 | 2 | 6 |
| B | 4 | 1 | 5 |
| C | 0 | 0 | 10 |

$$d_{\chi^2}(x, y) = \sum_i \frac{(x_i - y_i)^2}{x_i + y_i}$$

Usar la convención sin factor 1/2. Una casilla con ambos valores iguales a cero aporta 0.

a) Calcular las distancias de Q a las tres imágenes y determinar la más parecida.

b) Explicar por qué esta función resulta adecuada para comparar frecuencias de patrones y cómo pondera las diferencias entre casillas.

c) Una nueva imagen aporta 100 observaciones, diez veces las frecuencias de Q. ¿Qué ocurriría al usar los recuentos crudos? Indicar cómo normalizar todos los histogramas para comparar proporciones.

d) ¿Se puede aplicar automáticamente una poda basada en desigualdad triangular con esta disimilitud? Justificar.

Cada imagen se representa con un histograma de 3 casillas, con la cantidad de observaciones de cada patrón de textura. Se pierde dónde aparece cada patrón en la imagen. Uso chi cuadrado porque compara frecuencias casilla por casilla, y acá los patrones son categorías distintas sin una distancia entre ellas, así que no tiene sentido comparar una casilla con otra.

In [12]:
TEXTURAS = {"Q": (4, 0, 6), "A": (2, 2, 6), "B": (4, 1, 5), "C": (0, 0, 10)}


def aportes_chi(x, y):
    # una casilla con los dos valores en cero aporta 0
    return [(a - b) ** 2 / (a + b) if a + b != 0 else 0 for a, b in zip(x, y)]


def chi_cuadrado(x, y):
    return sum(aportes_chi(x, y))


d4 = {i: chi_cuadrado(TEXTURAS["Q"], TEXTURAS[i]) for i in "ABC"}

pd.DataFrame([[i] + aportes_chi(TEXTURAS["Q"], TEXTURAS[i]) + [d] for i, d in ordenar(d4)],
             columns=["Imagen", "Patrón 1", "Patrón 2", "Patrón 3", "Distancia"]).set_index("Imagen")

,Patrón 1,Patrón 2,Patrón 3,Distancia
Imagen,,,,
B,0.0000,1.0000,0.0909,1.0909
A,0.6667,2.0000,0.0000,2.6667
C,4.0000,0.0000,1.0000,5.0000


In [13]:
print("Imagen más parecida:", k_vecinos(d4, 1))

Imagen más parecida: ['B']


a) Cálculo completo para A: (4 − 2)² / (4 + 2) + (0 − 2)² / (0 + 2) + (6 − 6)² / (6 + 6) = 4/6 + 4/2 + 0 = 2.6667. La imagen más parecida a Q es B.

b) Sirve para frecuencias porque divide cada diferencia por el tamaño de la casilla. Una misma diferencia pesa más en una casilla con pocas observaciones que en una con muchas, porque en proporción es un cambio más grande. Se ve en A: la diferencia de 2 en el patrón 2 (0 contra 2) aporta 2.0000 y la misma diferencia de 2 en el patrón 1 (4 contra 2) aporta 0.6667. Así las casillas grandes no dominan la comparación.

In [14]:
def normalizar_histograma(h):
    total = sum(h)
    return tuple(v / total for v in h)


E = tuple(10 * v for v in TEXTURAS["Q"])
print("E:", E)
print("Con recuentos crudos, d(Q, E):", f4(chi_cuadrado(TEXTURAS["Q"], E)))

normalizados4 = {i: normalizar_histograma(h) for i, h in TEXTURAS.items()}
print("Normalizados, d(Q, E):", f4(chi_cuadrado(normalizados4["Q"], normalizar_histograma(E))))

dn4 = {i: chi_cuadrado(normalizados4["Q"], normalizados4[i]) for i in "ABC"}
tabla(dn4, "Distancia con histogramas normalizados")

E: (40, 0, 60)
Con recuentos crudos, d(Q, E): 73.6364
Normalizados, d(Q, E): 0.0000


,Distancia con histogramas normalizados
Objeto,
B,0.1091
A,0.2667
C,0.5000


c) Con los recuentos crudos, la imagen nueva E = (40, 0, 60) tiene las mismas proporciones que Q pero queda a distancia 73.6364, más lejos que todas las demás. La diferencia es solo la cantidad de observaciones. Para comparar proporciones hay que dividir cada histograma por su total de observaciones, de modo que sume 1. Q y E quedan los dos en (0.4, 0, 0.6) y su distancia es 0.0000. Todos los histogramas tienen que normalizarse igual. Las imágenes de 10 observaciones quedan a un décimo de su distancia: B 0.1091, A 0.2667 y C 0.5000, con el mismo orden.

In [15]:
d_QA = chi_cuadrado(TEXTURAS["Q"], TEXTURAS["A"])
d_QB = chi_cuadrado(TEXTURAS["Q"], TEXTURAS["B"])
d_BA = chi_cuadrado(TEXTURAS["B"], TEXTURAS["A"])
print("d(Q, A) =", f4(d_QA))
print("d(Q, B) + d(B, A) =", f4(d_QB), "+", f4(d_BA), "=", f4(d_QB + d_BA))
print("¿Se cumple la desigualdad triangular?", d_QA <= d_QB + d_BA)

d(Q, A) = 2.6667
d(Q, B) + d(B, A) = 1.0909 + 1.0909 = 2.1818
¿Se cumple la desigualdad triangular? False


d) No. Esta fórmula sin raíz no cumple la desigualdad triangular. Con los datos del ejercicio: d(Q, A) = 2.6667, pero d(Q, B) + d(B, A) = 1.0909 + 1.0909 = 2.1818. Una poda basada en esa desigualdad podría descartar objetos que sí están cerca. Habría que usar su raíz cuadrada, que sí es métrica, o no podar.

## Ejercicio 5. Wasserstein y distribución de intensidades

Un buscador de imágenes en escala de grises debe considerar pequeño un cambio hacia un nivel de intensidad vecino, y mayor un desplazamiento hacia un nivel lejano. Las imágenes se representan con histogramas normalizados sobre cuatro niveles igualmente espaciados: 0, 1, 2 y 3.

| Imagen | Nivel 0 | Nivel 1 | Nivel 2 | Nivel 3 |
|---|---|---|---|---|
| Q | 0 | 1 | 0 | 0 |
| A | 1 | 0 | 0 | 0 |
| B | 0 | 0 | 0 | 1 |
| C | 0 | 0.5 | 0.5 | 0 |

Utilizar Wasserstein de orden 1 en una dimensión. Cada unidad de masa transportada tiene costo igual a la distancia absoluta entre niveles. Todos los histogramas tienen masa total 1.

$$W_1(P, Q) = \sum_{i=0}^{2} |F_P(i) - F_Q(i)|$$

$F_P(i)$ es la masa acumulada desde el nivel 0 hasta el nivel i. El espaciado unitario ya está incorporado en la fórmula.

a) Calcular W1 desde Q hacia A, B y C, y ordenar las imágenes por similitud.

b) Describir un transporte de masa de costo mínimo para cada comparación. Relacionar ese costo con el cálculo por acumuladas.

c) Calcular la distancia Manhattan de Q a A y de Q a B. Explicar qué información sobre los niveles pierde esa comparación.

d) Si los niveles se ubican en 0, 10, 20 y 30, ¿cómo cambian las distancias Wasserstein? Indicar las unidades del resultado.

Cada imagen se representa con un histograma normalizado sobre cuatro niveles de intensidad. Se pierde dónde está cada píxel. Uso Wasserstein porque los niveles tienen un orden y una distancia entre ellos, y esta función tiene en cuenta cuán lejos hay que mover la masa. Eso es lo que pide el buscador: que un cambio a un nivel vecino cueste poco y a uno lejano cueste más.

In [16]:
INTENSIDADES = {"Q": (0, 1, 0, 0), "A": (1, 0, 0, 0), "B": (0, 0, 0, 1), "C": (0, 0.5, 0.5, 0)}


def acumulada(h):
    total = 0.0
    salida = []
    for v in h:
        total += v
        salida.append(total)
    return salida


def wasserstein(p, q, espaciado=1):
    # la última acumulada siempre es 1, por eso se suma hasta el nivel 2
    fp, fq = acumulada(p), acumulada(q)
    return espaciado * sum(abs(a - b) for a, b in zip(fp[:-1], fq[:-1]))


d5 = {i: wasserstein(INTENSIDADES["Q"], INTENSIDADES[i]) for i in "ABC"}

pd.DataFrame([["Q"] + acumulada(INTENSIDADES["Q"])[:-1] + [0.0]]
             + [[i] + acumulada(INTENSIDADES[i])[:-1] + [d] for i, d in ordenar(d5)],
             columns=["Imagen", "F(0)", "F(1)", "F(2)", "W1 con Q"]).set_index("Imagen")

,F(0),F(1),F(2),W1 con Q
Imagen,,,,
Q,0.0000,1.0000,1.0000,0.0000
C,0.0000,0.5000,1.0000,0.5000
A,1.0000,1.0000,1.0000,1.0000
B,0.0000,0.0000,0.0000,2.0000


a) Cálculo completo para C: las acumuladas son FQ = (0, 1, 1) y FC = (0, 0.5, 1), entonces W1(Q, C) = |0 − 0| + |1 − 0.5| + |1 − 1| = 0.5000. Por similitud el orden es C, A y B.

In [17]:
# cada movimiento es (masa, nivel de origen, nivel de destino)
TRANSPORTES = {"C": [(0.5, 1, 2)], "A": [(1, 1, 0)], "B": [(1, 1, 3)]}

pd.DataFrame([(i, f"mover {m} del nivel {o} al nivel {h}", sum(m * abs(o - h) for m, o, h in TRANSPORTES[i]), d5[i])
              for i, (m, o, h) in ((i, TRANSPORTES[i][0]) for i, _ in ordenar(d5))],
             columns=["Imagen", "Transporte desde Q", "Costo", "W1 por acumuladas"]).set_index("Imagen")

,Transporte desde Q,Costo,W1 por acumuladas
Imagen,,,
C,mover 0.5 del nivel 1 al nivel 2,0.5000,0.5000
A,mover 1 del nivel 1 al nivel 0,1.0000,1.0000
B,mover 1 del nivel 1 al nivel 3,2.0000,2.0000


b) |FP(i) − FQ(i)| es la masa que tiene que cruzar la frontera entre el nivel i y el i + 1. Cada cruce cuesta 1 por unidad de masa, así que sumar esas diferencias da el mismo costo que el transporte. Por ejemplo, de Q a B la masa cruza dos fronteras (entre 1 y 2, y entre 2 y 3), y por eso las acumuladas difieren en 1 para i = 1 y para i = 2.

In [18]:
def manhattan(x, y):
    return sum(abs(a - b) for a, b in zip(x, y))


print("Manhattan Q y A:", f4(manhattan(INTENSIDADES["Q"], INTENSIDADES["A"])))
print("Manhattan Q y B:", f4(manhattan(INTENSIDADES["Q"], INTENSIDADES["B"])))

Manhattan Q y A: 2.0000
Manhattan Q y B: 2.0000


c) Manhattan da 2.0000 para Q contra A y 2.0000 para Q contra B. Da lo mismo porque solo ve que la masa está en otra casilla, no cuán lejos. Pierde el orden de los niveles y la distancia entre ellos: pasar al nivel vecino o a uno más lejano es igual.

In [19]:
tabla({i: wasserstein(INTENSIDADES["Q"], INTENSIDADES[i], espaciado=10) for i in "ABC"},
      "W1 con niveles 0, 10, 20 y 30")

,"W1 con niveles 0, 10, 20 y 30"
Objeto,
C,5.0000
A,10.0000
B,20.0000


d) Se multiplican por 10, porque cada frontera entre niveles ahora cuesta 10: C 5.0000, A 10.0000 y B 20.0000. El orden no cambia. El resultado está en las mismas unidades que los niveles de intensidad, ya que la masa es una proporción y no tiene unidades. Se puede leer como cuánto hay que mover en promedio la intensidad de la imagen.

## Ejercicio 6. Canberra y componentes poco abundantes

Un laboratorio busca muestras con perfiles de concentración similares. Dos componentes abundantes y dos trazas se miden con el mismo protocolo. Las diferencias relativas en las trazas también importan, y no se desea que las concentraciones grandes dominen toda la comparación.

| Muestra | Componente 1 | Componente 2 | Traza 1 | Traza 2 |
|---|---|---|---|---|
| Q | 100 | 100 | 1 | 0 |
| A | 110 | 100 | 1 | 0 |
| B | 100 | 100 | 0 | 0 |
| C | 100 | 100 | 1 | 1 |

$$d_C(x, y) = \sum_i \frac{|x_i - y_i|}{|x_i| + |y_i|}$$

Los valores están en mg/L. Si ambos valores de una coordenada son cero, su aporte es 0.

a) Calcular las distancias a Q y recuperar todas las muestras con dC ≤ 0.1.

b) Comparar el aporte de pasar de 100 a 110 con el de pasar de 1 a 0. Interpretar por qué Canberra es pertinente para el objetivo del laboratorio.

c) Calcular el aporte de comparar 0 con 0.001. ¿Qué problema puede causar el ruido cercano al límite de detección? Proponer un criterio de preprocesamiento sin cambiar los datos de los incisos anteriores.

Cada muestra se representa con un vector de 4 concentraciones en mg/L. Uso Canberra porque divide cada diferencia por la suma de los dos valores, así cuenta la diferencia relativa en cada coordenada. Entonces las trazas pesan igual que los componentes abundantes, que es lo que pide el laboratorio.

In [20]:
MUESTRAS = {"Q": (100, 100, 1, 0), "A": (110, 100, 1, 0), "B": (100, 100, 0, 0), "C": (100, 100, 1, 1)}


def aportes_canberra(x, y):
    # si los dos valores son cero, el aporte es 0
    return [abs(a - b) / (abs(a) + abs(b)) if abs(a) + abs(b) != 0 else 0 for a, b in zip(x, y)]


def canberra(x, y):
    return sum(aportes_canberra(x, y))


d6 = {m: canberra(MUESTRAS["Q"], MUESTRAS[m]) for m in "ABC"}

pd.DataFrame([[m] + aportes_canberra(MUESTRAS["Q"], MUESTRAS[m]) + [d] for m, d in ordenar(d6)],
             columns=["Muestra", "Comp. 1", "Comp. 2", "Traza 1", "Traza 2", "dC"]).set_index("Muestra")

,Comp. 1,Comp. 2,Traza 1,Traza 2,dC
Muestra,,,,,
A,0.0476,0.0000,0.0000,0.0000,0.0476
B,0.0000,0.0000,1.0000,0.0000,1.0000
C,0.0000,0.0000,0.0000,1.0000,1.0000


In [21]:
print("Muestras con dC <= 0.1:", rango(d6, 0.1))

Muestras con dC <= 0.1: ['A']


a) Cálculo completo para A: |100 − 110| / (100 + 110) + 0/200 + 0/2 + 0 = 10/210 = 0.0476. La última coordenada aporta 0 porque los dos valores son cero. B y C empatan en 1.0000 y van en orden alfabético. Con dC menor o igual que 0.1 se recupera solo A.

In [22]:
print("De 100 a 110:", f4(aportes_canberra((100,), (110,))[0]))
print("De 1 a 0:", f4(aportes_canberra((1,), (0,))[0]))
print("Manhattan Q y A:", f4(manhattan(MUESTRAS["Q"], MUESTRAS["A"])))
print("Manhattan Q y B:", f4(manhattan(MUESTRAS["Q"], MUESTRAS["B"])))

De 100 a 110: 0.0476
De 1 a 0: 1.0000
Manhattan Q y A: 10.0000
Manhattan Q y B: 1.0000


b) Pasar de 100 a 110 aporta 10/210 = 0.0476. Pasar de 1 a 0 aporta 1/1 = 1.0000, que es lo máximo que puede aportar una coordenada. La diferencia absoluta del primer caso es diez veces mayor, pero en proporción es mucho menor. Por eso Canberra es pertinente: se nota si desaparece una traza, y 10 mg/L más sobre 100 casi no cambia el perfil. Con Manhattan sería al revés: A quedaría a 10 y B a 1.

In [23]:
def aplicar_limite(muestra, limite):
    # todo valor por debajo del límite de detección se toma como 0
    return tuple(v if v >= limite else 0 for v in muestra)


LIMITE = 0.01
print("0 contra 0.001:", f4(aportes_canberra((0,), (0.001,))[0]))
print("0 contra 0.001 con el límite:", f4(aportes_canberra(aplicar_limite((0,), LIMITE), aplicar_limite((0.001,), LIMITE))[0]))

# los datos de los incisos anteriores no cambian
print("Muestras sin cambios:", all(aplicar_limite(m, LIMITE) == m for m in MUESTRAS.values()))

0 contra 0.001: 1.0000
0 contra 0.001 con el límite: 0.0000
Muestras sin cambios: True


c) Comparar 0 con 0.001 aporta 0.001/0.001 = 1.0000, lo mismo que perder una traza entera. Si 0.001 es ruido cerca del límite de detección, una medición casi nula le suma 1 a la distancia y la muestra queda lejos sin ninguna razón química. Como criterio propongo definir un límite de detección según el protocolo del laboratorio (por ejemplo 0.01 mg/L) y tomar como 0 todo valor por debajo de ese límite en todas las muestras antes de calcular. Los datos de los incisos anteriores no cambian porque ningún valor distinto de cero está por debajo del límite, y 0 contra 0.001 pasa a aportar 0.

## Ejercicio 7. Bray Curtis y comunidades de aves

Un equipo de ecología compara sitios muestreados durante el mismo tiempo y sobre la misma superficie. Busca comunidades con abundancias semejantes, dando mayor influencia a las especies abundantes. Las ausencias compartidas no deben aumentar artificialmente la similitud.

| Sitio | Gorriones | Tordos | Horneros | Calandrias |
|---|---|---|---|---|
| Q | 8 | 2 | 0 | 0 |
| A | 6 | 4 | 0 | 0 |
| B | 0 | 0 | 8 | 2 |
| C | 16 | 4 | 0 | 0 |

$$d_{BC}(x, y) = \frac{\sum_i |x_i - y_i|}{\sum_i (x_i + y_i)}$$

a) Calcular las disimilitudes a Q y obtener los 2 sitios más similares.

b) Q y C tienen las mismas proporciones. Explicar por qué su disimilitud sobre recuentos no es cero.

c) Normalizar cada sitio para que sus abundancias sumen 1 y recalcular Q contra C. Explicar cuándo conviene comparar composición relativa y cuándo abundancias absolutas.

d) ¿Qué cambia si se añade una especie ausente en todos los sitios? ¿Qué decisión sería necesaria si ambas muestras estuvieran vacías?

Cada sitio se representa con un vector de abundancias, la cantidad de individuos de cada especie. Se pueden comparar recuentos directamente porque el muestreo fue en el mismo tiempo y superficie. Uso Bray-Curtis porque mide cuánta abundancia no coincide sobre el total: las especies abundantes influyen más y las ausencias compartidas no cuentan, porque suman 0 arriba y abajo.

In [24]:
SITIOS = {"Q": (8, 2, 0, 0), "A": (6, 4, 0, 0), "B": (0, 0, 8, 2), "C": (16, 4, 0, 0)}


def bray_curtis(x, y):
    denominador = sum(a + b for a, b in zip(x, y))
    if denominador == 0:
        return None  # dos muestras vacías, sin datos para comparar
    return sum(abs(a - b) for a, b in zip(x, y)) / denominador


d7 = {s: bray_curtis(SITIOS["Q"], SITIOS[s]) for s in "ABC"}

pd.DataFrame([(s, sum(abs(a - b) for a, b in zip(SITIOS["Q"], SITIOS[s])),
               sum(a + b for a, b in zip(SITIOS["Q"], SITIOS[s])), d) for s, d in ordenar(d7)],
             columns=["Sitio", "Suma de diferencias", "Suma total", "dBC"]).set_index("Sitio")

,Suma de diferencias,Suma total,dBC
Sitio,,,
A,4,20,0.2000
C,10,30,0.3333
B,20,20,1.0000


In [25]:
print("2 sitios más similares:", k_vecinos(d7, 2))

2 sitios más similares: ['A', 'C']


a) Cálculo completo para A: (|8 − 6| + |2 − 4| + 0 + 0) / (14 + 6 + 0 + 0) = 4/20 = 0.2000. Los 2 sitios más similares a Q son A y C.

b) Sobre recuentos la disimilitud también mide la cantidad total, no solo las proporciones. C tiene el doble de individuos que Q, y esos 10 individuos de más son la diferencia: 10/30 = 0.3333. No es cero porque en C hay más aves.

In [26]:
normalizados7 = {s: tuple(v / sum(h) for v in h) for s, h in SITIOS.items()}
print("Q normalizado:", normalizados7["Q"])
print("C normalizado:", normalizados7["C"])
print("dBC(Q, C) normalizados:", f4(bray_curtis(normalizados7["Q"], normalizados7["C"])))

Q normalizado: (0.8, 0.2, 0.0, 0.0)
C normalizado: (0.8, 0.2, 0.0, 0.0)
dBC(Q, C) normalizados: 0.0000


c) Normalizados, Q y C quedan los dos en (0.8, 0.2, 0, 0), así que su disimilitud es 0/2 = 0.0000. Conviene comparar composición relativa cuando interesa qué especies dominan y la cantidad total depende del esfuerzo de muestreo, por ejemplo con muestreos de distinto tiempo o superficie. Conviene comparar abundancias absolutas cuando el muestreo es el mismo y la cantidad de individuos también es información, como acá, donde C tiene el doble de aves.

In [27]:
con_especie = {s: h + (0,) for s, h in SITIOS.items()}
print("Dos muestras vacías:", bray_curtis((0, 0), (0, 0)))
tabla({s: bray_curtis(con_especie["Q"], con_especie[s]) for s in "ABC"}, "dBC con una especie ausente en todos")

Dos muestras vacías: None


,dBC con una especie ausente en todos
Objeto,
A,0.2000
C,0.3333
B,1.0000


d) Si se agrega una especie ausente en todos los sitios no cambia nada: suma 0 en el numerador y en el denominador. Si dos muestras estuvieran vacías queda 0/0 y hay que decidir qué hacer, porque la fórmula no está definida. Decidiría no compararlas y marcarlas como sin datos, porque no hay ninguna comunidad que comparar. Si el sistema necesitara un número, tomaría d = 0 porque son iguales.

## Ejercicio 8. KL y Jensen Shannon en perfiles de noticias

Un portal representa la distribución de lecturas por dos temas: tecnología y deportes. P = (0.5, 0.5) es el perfil de un lector. Los perfiles candidatos son A = (0.9, 0.1), B = (0.6, 0.4) y C = (1, 0). Se desea comparar perfiles probabilísticos, teniendo en cuenta el sentido de la comparación cuando corresponda.

$$D_{KL}(P \| Q) = \sum_i P_i \ln\frac{P_i}{Q_i}$$

$$M = \frac{P + Q}{2} \qquad JS(P, Q) = \frac{D_{KL}(P \| M) + D_{KL}(Q \| M)}{2} \qquad d_{JS}(P, Q) = \sqrt{JS(P, Q)}$$

Usar logaritmos naturales. Un término con $P_i = 0$ aporta 0. Si $P_i > 0$ y $Q_i = 0$, KL es infinita. JS denota la divergencia, dJS su raíz cuadrada.

a) Calcular DKL(P ‖ A) y DKL(A ‖ P). Interpretar por qué no son intercambiables cuando un perfil se usa para aproximar al otro.

b) Calcular JS(P, Q) y dJS(P, Q) para cada candidato Q ∈ {A, B, C}. Ordenarlos por cercanía al lector P.

c) Explicar por qué DKL(P ‖ C) es infinita y JS(P, C) es finita. Mostrar el promedio M utilizado para el candidato C.

d) Para recuperar perfiles similares sin privilegiar una dirección, ¿cuál de estas medidas elegiría? Si además se exige una métrica, precisar cuál corresponde y por qué.

Cada lector o perfil se representa con una distribución de probabilidad sobre dos temas. Se pierde cuánto lee en total y queda solo la proporción. Uso KL cuando importa el sentido (un perfil que aproxima a otro) y JS cuando se quiere comparar sin dirección.

In [28]:
P = (0.5, 0.5)
PERFILES = {"A": (0.9, 0.1), "B": (0.6, 0.4), "C": (1, 0)}


def kl(p, q):
    total = 0
    for a, b in zip(p, q):
        if a == 0:
            continue  # un término con P = 0 aporta 0
        if b == 0:
            return math.inf  # P > 0 y Q = 0
        total += a * math.log(a / b)
    return total


def promedio(p, q):
    return tuple((a + b) / 2 for a, b in zip(p, q))


def js(p, q):
    m = promedio(p, q)
    return (kl(p, m) + kl(q, m)) / 2


def d_js(p, q):
    return math.sqrt(js(p, q))


print("DKL(P || A) =", f4(kl(P, PERFILES["A"])))
print("DKL(A || P) =", f4(kl(PERFILES["A"], P)))

DKL(P || A) = 0.5108
DKL(A || P) = 0.3681


a) DKL(P ‖ A) = 0.5 ln(0.5/0.9) + 0.5 ln(0.5/0.1) = −0.2939 + 0.8047 = 0.5108. DKL(A ‖ P) = 0.9 ln(0.9/0.5) + 0.1 ln(0.1/0.5) = 0.5290 − 0.1609 = 0.3681. No son intercambiables porque KL mide lo que se pierde al usar el segundo perfil para aproximar al primero, y pesa cada término según el primero. En DKL(P ‖ A) se usa A para aproximar al lector: el lector lee deportes la mitad del tiempo y A le da solo 0.1, y eso se penaliza. En DKL(A ‖ P) los pesos los pone A, que casi no tiene deportes, y ese término pesa poco.

In [29]:
d8 = {c: d_js(P, PERFILES[c]) for c in "ABC"}

pd.DataFrame([(c, tuple(round(v, 4) for v in promedio(P, PERFILES[c])), kl(P, promedio(P, PERFILES[c])),
               kl(PERFILES[c], promedio(P, PERFILES[c])), js(P, PERFILES[c]), d) for c, d in ordenar(d8)],
             columns=["Candidato", "M", "DKL(P || M)", "DKL(Q || M)", "JS", "dJS"]).set_index("Candidato")

,M,DKL(P || M),DKL(Q || M),JS,dJS
Candidato,,,,,
B,"(0.55, 0.45)",0.0050,0.0051,0.0051,0.0711
A,"(0.7, 0.3)",0.0872,0.1163,0.1017,0.3190
C,"(0.75, 0.25)",0.1438,0.2877,0.2158,0.4645


b) Cálculo completo para A: M = (0.7, 0.3). DKL(P ‖ M) = 0.0872 y DKL(A ‖ M) = 0.1163, entonces JS(P, A) = (0.0872 + 0.1163) / 2 = 0.1017 y dJS(P, A) = √0.1017 = 0.3190. Por cercanía al lector el orden es B, A y C.

In [30]:
print("DKL(P || C) =", kl(P, PERFILES["C"]))
print("M para C:", promedio(P, PERFILES["C"]))
print("JS(P, C) =", f4(js(P, PERFILES["C"])))

DKL(P || C) = inf
M para C: (0.75, 0.25)
JS(P, C) = 0.2158


c) DKL(P ‖ C) es infinita porque P tiene 0.5 en deportes y C tiene 0, y el término 0.5 ln(0.5/0) no está acotado: C no puede explicar algo que el lector sí lee. JS no tiene ese problema porque cada perfil se compara con el promedio M, y M tiene masa donde al menos uno de los dos la tiene. Para C, M = (0.75, 0.25). DKL(P ‖ M) = 0.1438 y DKL(C ‖ M) = 0.2877, porque el término con C = 0 aporta 0. Entonces JS(P, C) = 0.2158.

In [31]:
b, c = PERFILES["B"], PERFILES["C"]
print("JS(P, C) =", f4(js(P, c)))
print("JS(P, B) + JS(B, C) =", f4(js(P, b)), "+", f4(js(b, c)), "=", f4(js(P, b) + js(b, c)))
print("dJS(P, C) =", f4(d_js(P, c)))
print("dJS(P, B) + dJS(B, C) =", f4(d_js(P, b) + d_js(b, c)))

JS(P, C) = 0.2158
JS(P, B) + JS(B, C) = 0.0051 + 0.1639 = 0.1690
dJS(P, C) = 0.4645
dJS(P, B) + dJS(B, C) = 0.4760


d) Elegiría JS, porque es simétrica y siempre finita. Si además se exige una métrica, corresponde dJS, la raíz cuadrada de JS, que sí cumple la desigualdad triangular. JS sola no la cumple: JS(P, C) = 0.2158, pero JS(P, B) + JS(B, C) = 0.0051 + 0.1639 = 0.1690. Con la raíz sí se cumple: dJS(P, C) = 0.4645 es menor que dJS(P, B) + dJS(B, C) = 0.4760. KL tampoco es una métrica y además no es simétrica.

## Ejercicio 9. Hausdorff y comparación de contornos

Un sistema de inspección compara contornos simplificados de piezas, previamente alineados y expresados en milímetros. Cada contorno es un conjunto de puntos sin orden. Se busca controlar la peor separación entre el contorno observado y el de referencia.

Referencia Q = {(0, 0), (2, 0), (2, 2)}.

Pieza A = {(0, 0), (2, 0), (2, 3)}.

Pieza B = {(0, 0), (2, 0), (2, 2), (8, 8)}.

$$h(A, B) = \max_{a \in A} \min_{b \in B} \|a - b\|_2 \qquad d_H(A, B) = \max\{h(A, B), h(B, A)\}$$

a) Para Q y A, mostrar la distancia mínima desde cada punto al otro conjunto. Calcular ambas distancias dirigidas y la distancia de Hausdorff.

b) Repetir el procedimiento para Q y B. Explicar por qué no alcanza con calcular solamente h(Q, B).

c) Recuperar las piezas a distancia de Hausdorff menor o igual que 1 mm de Q.

d) Si (8, 8) fuera ruido del sensor, ¿cómo afectaría la decisión? Proponer un tratamiento o una variante robusta, e indicar qué se pierde si ese punto fuera un defecto real.

Cada contorno se representa como un conjunto de puntos en milímetros, sin orden. Uso Hausdorff porque toma la peor separación entre los dos contornos, que es lo que se quiere controlar en la inspección.

In [32]:
CONTORNOS = {"Q": [(0, 0), (2, 0), (2, 2)],
             "A": [(0, 0), (2, 0), (2, 3)],
             "B": [(0, 0), (2, 0), (2, 2), (8, 8)]}


def minimos(a, b):
    # distancia de cada punto de a al punto más cercano de b
    return [min(euclidiana(p, r) for r in b) for p in a]


def h(a, b):
    return max(minimos(a, b))


def hausdorff(a, b):
    return max(h(a, b), h(b, a))


def tabla_minimos(nombre_a, nombre_b):
    a, b = CONTORNOS[nombre_a], CONTORNOS[nombre_b]
    return pd.DataFrame([(f"{nombre_a} a {nombre_b}", p, d) for p, d in zip(a, minimos(a, b))],
                        columns=["Sentido", "Punto", "Distancia mínima"])


pd.concat([tabla_minimos("Q", "A"), tabla_minimos("A", "Q")], ignore_index=True)

,Sentido,Punto,Distancia mínima
0,Q a A,"(0, 0)",0.0000
1,Q a A,"(2, 0)",0.0000
2,Q a A,"(2, 2)",1.0000
3,A a Q,"(0, 0)",0.0000
4,A a Q,"(2, 0)",0.0000
5,A a Q,"(2, 3)",1.0000


In [33]:
print("h(Q, A) =", f4(h(CONTORNOS["Q"], CONTORNOS["A"])))
print("h(A, Q) =", f4(h(CONTORNOS["A"], CONTORNOS["Q"])))
print("dH(Q, A) =", f4(hausdorff(CONTORNOS["Q"], CONTORNOS["A"])))

h(Q, A) = 1.0000
h(A, Q) = 1.0000
dH(Q, A) = 1.0000


a) Cálculo completo para el punto (2, 2) de Q: a (0, 0) hay √8 = 2.8284, a (2, 0) hay 2 y a (2, 3) hay 1, así que su mínimo es 1. Entonces h(Q, A) = 1.0000, h(A, Q) = 1.0000 y dH(Q, A) = 1.0000.

In [34]:
pd.concat([tabla_minimos("Q", "B"), tabla_minimos("B", "Q")], ignore_index=True)

,Sentido,Punto,Distancia mínima
0,Q a B,"(0, 0)",0.0000
1,Q a B,"(2, 0)",0.0000
2,Q a B,"(2, 2)",0.0000
3,B a Q,"(0, 0)",0.0000
4,B a Q,"(2, 0)",0.0000
5,B a Q,"(2, 2)",0.0000
6,B a Q,"(8, 8)",8.4853


In [35]:
print("h(Q, B) =", f4(h(CONTORNOS["Q"], CONTORNOS["B"])))
print("h(B, Q) =", f4(h(CONTORNOS["B"], CONTORNOS["Q"])))
print("dH(Q, B) =", f4(hausdorff(CONTORNOS["Q"], CONTORNOS["B"])))

h(Q, B) = 0.0000
h(B, Q) = 8.4853
dH(Q, B) = 8.4853


b) Para Q y B, h(Q, B) = 0.0000, h(B, Q) = 8.4853 y dH(Q, B) = 8.4853. El valor sale de (8, 8) contra (2, 2): √(6² + 6²) = √72. No alcanza con h(Q, B) porque la distancia dirigida solo mira si cada punto de Q tiene uno cerca en B. Da 0 aunque B tenga un punto de más muy lejos. Hay que mirar los dos sentidos.

In [36]:
d9 = {p: hausdorff(CONTORNOS["Q"], CONTORNOS[p]) for p in "AB"}
print("Piezas a 1 mm o menos:", rango(d9, 1))
tabla(d9, "dH con Q")

Piezas a 1 mm o menos: ['A']


,dH con Q
Objeto,
A,1.0000
B,8.4853


c) Con distancia menor o igual que 1 mm se recupera solo A, que está justo en el límite.

In [37]:
def h_parcial(a, b, percentil):
    # en lugar del máximo toma un percentil de las distancias mínimas
    valores = sorted(minimos(a, b))
    posicion = math.ceil(percentil / 100 * len(valores)) - 1
    return valores[posicion]


print("Distancias mínimas de B a Q:", [f4(v) for v in sorted(minimos(CONTORNOS["B"], CONTORNOS["Q"]))])
print("h parcial con percentil 75:", f4(h_parcial(CONTORNOS["B"], CONTORNOS["Q"], 75)))

Distancias mínimas de B a Q: ['0.0000', '0.0000', '0.0000', '8.4853']
h parcial con percentil 75: 0.0000


d) Si (8, 8) fuera ruido, B se rechazaría aunque sus otros tres puntos coinciden exactamente con Q, porque Hausdorff toma el máximo y depende de un solo punto. Propongo repetir la medición o filtrar puntos aislados antes de calcular. Otra opción es la Hausdorff parcial, que en lugar del máximo toma un percentil de las distancias mínimas: con el percentil 75, h(B, Q) pasa a 0, porque las distancias son 0, 0, 0 y 8.4853 y se toma la tercera. Lo que se pierde es que, si (8, 8) fuera un defecto real, como una rebaba, la variante robusta no lo detecta y la pieza pasa el control. Eso va en contra de controlar la peor separación, por eso repetiría la medición antes de descartar el punto.

## Ejercicio 10. DTW y gestos realizados a distinto ritmo

Una pulsera registra una característica de movimiento ya normalizada. Se buscan gestos parecidos aunque duren distinta cantidad de muestras. Q = (0, 1, 2, 1, 0) es la referencia, A = (0, 1, 1, 2, 1, 0) contiene una fase más lenta y B = (0, 1, 0, 1, 0) describe otro movimiento.

Usar costo local $|x_i - y_j|$ y costo acumulado sin normalizar por la longitud del camino. Se permiten pasos horizontal, vertical y diagonal, sin ventana de restricción.

$$D(i, j) = |x_i - y_j| + \min\{D(i-1, j), D(i, j-1), D(i-1, j-1)\}$$

Inicializar $D(0, 0) = 0$, $D(i, 0) = \infty$ para $i > 0$ y $D(0, j) = \infty$ para $j > 0$. $DTW(x, y) = D(n, m)$.

a) Construir las matrices acumuladas para Q contra A y Q contra B. Obtener las distancias y elegir el gesto más parecido.

b) Reconstruir un camino óptimo para Q contra A e indicar qué muestras se repiten en el alineamiento.

c) Explicar por qué la comparación punto a punto no resuelve directamente la diferencia de duración. Si dos series distintas tienen DTW igual a cero, ¿qué propiedad métrica se incumple?

d) Si la duración fuera relevante para distinguir gestos, ¿qué restricción o información adicional incorporaría?

Cada gesto se representa con una serie temporal, la secuencia de valores de la característica de movimiento. Se conservan la forma y el orden, y al alinear se deja de lado la duración. Uso DTW porque alinea las series aunque tengan distinta cantidad de muestras, y A es el mismo gesto que Q con una fase más lenta.

In [38]:
GESTOS = {"Q": (0, 1, 2, 1, 0), "A": (0, 1, 1, 2, 1, 0), "B": (0, 1, 0, 1, 0)}


def matriz_dtw(x, y):
    n, m = len(x), len(y)
    D = [[math.inf] * (m + 1) for _ in range(n + 1)]
    D[0][0] = 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            costo = abs(x[i - 1] - y[j - 1])
            D[i][j] = costo + min(D[i - 1][j], D[i][j - 1], D[i - 1][j - 1])
    return D


def ver_matriz(x, y, D):
    filas = [f"{i} ({x[i - 1]})" if i > 0 else "0" for i in range(len(x) + 1)]
    columnas = [f"{j} ({y[j - 1]})" if j > 0 else "0" for j in range(len(y) + 1)]
    valores = [["∞" if v == math.inf else int(v) for v in fila] for fila in D]
    return pd.DataFrame(valores, index=filas, columns=columnas)


D_QA = matriz_dtw(GESTOS["Q"], GESTOS["A"])
ver_matriz(GESTOS["Q"], GESTOS["A"], D_QA)

,0,1 (0),2 (1),3 (1),4 (2),5 (1),6 (0)
0,0,∞,∞,∞,∞,∞,∞
1 (0),∞,0,1,2,4,5,5
2 (1),∞,1,0,0,1,1,2
3 (2),∞,3,1,1,0,1,3
4 (1),∞,4,1,1,1,0,1
5 (0),∞,4,2,2,3,1,0


In [39]:
D_QB = matriz_dtw(GESTOS["Q"], GESTOS["B"])
ver_matriz(GESTOS["Q"], GESTOS["B"], D_QB)

,0,1 (0),2 (1),3 (0),4 (1),5 (0)
0,0,∞,∞,∞,∞,∞
1 (0),∞,0,1,1,2,2
2 (1),∞,1,0,1,1,2
3 (2),∞,3,1,2,2,3
4 (1),∞,4,1,2,2,3
5 (0),∞,4,2,1,2,2


In [40]:
d10 = {"A": float(D_QA[-1][-1]), "B": float(D_QB[-1][-1])}
tabla(d10, "DTW con Q")

,DTW con Q
Objeto,
A,0.0000
B,2.0000


a) Cálculo completo de una celda de Q contra A: D(2, 2) = |1 − 1| + min{D(1, 2), D(2, 1), D(1, 1)} = 0 + min{1, 1, 0} = 0. El resultado es DTW(Q, A) = 0.0000 y DTW(Q, B) = 2.0000, así que el gesto más parecido es A.

In [41]:
def camino_optimo(x, y, D):
    # se reconstruye desde el final, prefiriendo la diagonal
    i, j = len(x), len(y)
    camino = [(i, j)]
    while (i, j) != (1, 1):
        anterior = D[i][j] - abs(x[i - 1] - y[j - 1])
        if D[i - 1][j - 1] == anterior:
            i, j = i - 1, j - 1
        elif D[i][j - 1] == anterior:
            j = j - 1
        else:
            i = i - 1
        camino.append((i, j))
    return camino[::-1]


camino = camino_optimo(GESTOS["Q"], GESTOS["A"], D_QA)
pd.DataFrame([(i, GESTOS["Q"][i - 1], j, GESTOS["A"][j - 1], abs(GESTOS["Q"][i - 1] - GESTOS["A"][j - 1]))
              for i, j in camino],
             columns=["Muestra de Q", "Valor en Q", "Muestra de A", "Valor en A", "Costo local"])

,Muestra de Q,Valor en Q,Muestra de A,Valor en A,Costo local
0,1,0,1,0,0
1,2,1,2,1,0
2,2,1,3,1,0
3,3,2,4,2,0
4,4,1,5,1,0
5,5,0,6,0,0


b) Un camino óptimo para Q contra A es (1, 1), (2, 2), (2, 3), (3, 4), (4, 5), (5, 6), con costo total 0. La muestra 2 de Q, que vale 1, se repite: se alinea con las muestras 2 y 3 de A.

In [42]:
print("Punto a punto Q y primeras 5 muestras de A:", f4(manhattan(GESTOS["Q"], GESTOS["A"][:5])))
print("Punto a punto Q y B:", f4(manhattan(GESTOS["Q"], GESTOS["B"])))
print("DTW(Q, A):", f4(d10["A"]), "y Q == A:", GESTOS["Q"] == GESTOS["A"])

Punto a punto Q y primeras 5 muestras de A: 3.0000
Punto a punto Q y B: 2.0000
DTW(Q, A): 0.0000 y Q == A: False


c) La comparación punto a punto necesita la misma cantidad de muestras, y Q tiene 5 y A tiene 6. Aunque se recorte A, desde la fase lenta en adelante todo queda corrido: con las primeras 5 muestras de A la suma de diferencias da 3, aunque es el mismo gesto. Si dos series distintas tienen DTW igual a cero, como Q y A, se incumple la identidad: en una métrica d(x, y) = 0 solo si x = y.

d) Agregaría una banda de Sakoe-Chiba, que es una ventana que limita cuánto se puede alejar el camino de la diagonal, o una penalización por cada paso horizontal o vertical. También guardaría la duración como un dato aparte y exigiría que la diferencia de duración no pase de cierto valor.

## Actividad complementaria. Coseno en búsqueda de documentos

Esta función ya se trabajó en el TP I. La siguiente actividad permite revisarla en un contexto donde interesa comparar el tema de documentos de distinta extensión.

Una biblioteca representa documentos mediante frecuencias de tres términos, en este orden: algoritmo, datos, red. Se ignoran el orden de las palabras y los demás términos. La consulta es q = (1, 1, 0).

| Documento | algoritmo | datos | red |
|---|---|---|---|
| A | 1 | 1 | 0 |
| B | 10 | 10 | 0 |
| C | 1 | 0 | 1 |
| D | 0 | 0 | 5 |

$$d_{cos}(x, y) = 1 - \frac{x \cdot y}{\|x\|_2 \|y\|_2}$$

a) Calcular las cuatro distancias y devolver los 2 documentos más cercanos.

b) Interpretar el resultado de A y B. Explicar por qué es útil que la longitud del vector no determine por sí sola la similitud temática.

c) Un documento E queda representado como (0, 0, 0). Explicar por qué la fórmula no está definida y proponer cómo tratarlo en el sistema.

d) ¿Una distancia coseno igual a cero implica que dos documentos son idénticos? ¿Puede utilizarse sin más en un índice que exige una métrica? Fundamentar ambas respuestas.

Cada documento se representa con un vector de frecuencias de tres términos. Se pierden el orden de las palabras y los demás términos. Uso coseno porque compara la dirección de los vectores, es decir las proporciones entre términos, sin importar la extensión del documento.

In [43]:
Q_DOC = (1, 1, 0)
DOCUMENTOS = {"A": (1, 1, 0), "B": (10, 10, 0), "C": (1, 0, 1), "D": (0, 0, 5)}


def coseno(x, y):
    norma_x = math.sqrt(sum(a * a for a in x))
    norma_y = math.sqrt(sum(b * b for b in y))
    if norma_x == 0 or norma_y == 0:
        raise ValueError("La distancia coseno no está definida para un vector nulo")
    return 1 - sum(a * b for a, b in zip(x, y)) / (norma_x * norma_y)


dc = {i: coseno(Q_DOC, v) for i, v in DOCUMENTOS.items()}
print("2 documentos más cercanos:", k_vecinos(dc, 2))
tabla(dc, "Distancia coseno")

2 documentos más cercanos: ['A', 'B']


,Distancia coseno
Objeto,
A,0.0000
B,0.0000
C,0.5000
D,1.0000


a) Cálculo completo para C: el producto interno de q y C es 1, ‖q‖ = √2 y ‖C‖ = √2, entonces dcos(q, C) = 1 − 1/2 = 0.5000. Los 2 documentos más cercanos son A y B.

In [44]:
print("Euclidiana q y B:", f4(euclidiana(Q_DOC, DOCUMENTOS["B"])))
print("Euclidiana q y C:", f4(euclidiana(Q_DOC, DOCUMENTOS["C"])))

Euclidiana q y B: 12.7279
Euclidiana q y C: 1.4142


b) A y B dan 0 porque tienen la misma proporción de términos: B repite diez veces lo de A. Hablan de lo mismo y B es simplemente más largo. Es útil que la longitud no decida porque, si contara, un documento largo sobre el tema quedaría lejos: con la euclidiana B queda a 12.7279 de q y C, que trata de otra cosa, a 1.4142.

In [45]:
try:
    coseno(Q_DOC, (0, 0, 0))
except ValueError as error:
    print("Documento E:", error)

Documento E: La distancia coseno no está definida para un vector nulo


c) E tiene norma 0 y la fórmula divide por la norma, así que queda una división por cero. Un vector nulo no tiene dirección. Lo trataría como un documento sin términos: no lo indexo y no lo devuelvo en las búsquedas. Si hace falta un valor, le asigno la distancia máxima, 1.

In [46]:
d_qD = coseno(Q_DOC, DOCUMENTOS["D"])
d_qC = coseno(Q_DOC, DOCUMENTOS["C"])
d_CD = coseno(DOCUMENTOS["C"], DOCUMENTOS["D"])
print("d(q, D) =", f4(d_qD))
print("d(q, C) + d(C, D) =", f4(d_qC), "+", f4(d_CD), "=", f4(d_qC + d_CD))
print("¿Se cumple la desigualdad triangular?", d_qD <= d_qC + d_CD)

d(q, D) = 1.0000
d(q, C) + d(C, D) = 0.5000 + 0.2929 = 0.7929
¿Se cumple la desigualdad triangular? False


d) No implica que sean idénticos. Distancia 0 solo dice que los vectores son proporcionales: A y B dan 0 y tienen distintas frecuencias. Incluso con el mismo vector los documentos podrían ser distintos, porque se ignoran el orden y los demás términos. Tampoco se puede usar sin más en un índice métrico, porque no es métrica: no cumple la identidad ni la desigualdad triangular. Por ejemplo d(q, D) = 1.0000, pero d(q, C) + d(C, D) = 0.5000 + 0.2929 = 0.7929. Para usarla habría que normalizar los vectores a largo 1 y pasar a la distancia angular (el arco coseno de la similitud), que sí cumple la desigualdad triangular.

## Cierre integrador

Elegir tres ejercicios correspondientes a distintos tipos de datos. Para cada uno, redactar una respuesta que conecte las siguientes decisiones:

a) Qué objeto real se busca y qué información conserva o descarta su representación.

b) Qué significa que dos objetos sean parecidos y por qué la función elegida refleja esa noción.

c) Qué función alternativa de la presentación podría producir un resultado poco útil, y por qué.

d) Qué consulta necesita la aplicación: vecino más cercano, k vecinos o rango. Precisar qué representa el umbral si se elige rango.

In [47]:
# Valores que se citan en el cierre
print("Chi cuadrado Q y A del ejercicio 5:", f4(chi_cuadrado(INTENSIDADES["Q"], INTENSIDADES["A"])))
print("Chi cuadrado Q y B del ejercicio 5:", f4(chi_cuadrado(INTENSIDADES["Q"], INTENSIDADES["B"])))
print("Gestos con DTW menor o igual que 1:", rango(d10, 1))

Chi cuadrado Q y A del ejercicio 5: 2.0000
Chi cuadrado Q y B del ejercicio 5: 2.0000
Gestos con DTW menor o igual que 1: ['A']


### Ejercicio 1: conjuntos, con Jaccard

a) Se buscan clientes con hábitos de compra parecidos. La representación conserva qué categorías compró cada uno en el mes y descarta las cantidades, los precios y el orden.

b) Dos clientes son parecidos si gran parte de lo que compraron es lo mismo. Jaccard lo refleja porque compara lo compartido con el total que compraron los dos, y no cuenta lo que ninguno compró.

c) Manhattan sobre las unidades compradas daría un resultado poco útil: un cliente que compra diez veces más de lo mismo quedaría muy lejos, aunque tenga los mismos hábitos.

d) Necesita k vecinos, porque para sugerir productos alcanza con una cantidad fija de clientes parecidos, como los 2 de este ejercicio.

### Ejercicio 5: histogramas, con Wasserstein

a) Se buscan imágenes en escala de grises con intensidades parecidas. El histograma conserva qué proporción de la imagen está en cada nivel y descarta dónde está cada píxel, así que dos imágenes distintas con el mismo histograma quedan a distancia 0.

b) Dos imágenes son parecidas si se puede pasar de una a otra moviendo poca masa a niveles cercanos. Wasserstein mide justo ese costo.

c) Chi cuadrado o Manhattan comparan casilla por casilla y no ven la distancia entre niveles. Las dos dan 2.0000 para Q contra A y para Q contra B, aunque A está a un nivel de distancia y B a dos.

d) Un buscador necesita k vecinos, para mostrar las imágenes más parecidas en orden, haya o no alguna muy cercana.

### Ejercicio 10: series temporales, con DTW

a) Se buscan gestos parecidos registrados por una pulsera. La serie conserva la forma y el orden del movimiento, y al alinear con DTW se deja de lado el ritmo.

b) Dos gestos son parecidos si siguen la misma forma, aunque uno sea más lento. DTW lo refleja porque estira o comprime el tiempo para alinear las series.

c) Una comparación punto a punto como Manhattan pide la misma longitud y penaliza el desfase. Recortando A a 5 muestras daría 3.0000 contra Q y B daría 2.0000, así que elegiría B, que es otro movimiento.

d) Necesita una consulta de rango: el gesto se reconoce solo si está suficientemente cerca de la referencia, y si no hay ninguno no se reconoce nada. El umbral representa el costo acumulado de desalineación que se tolera, en unidades de la característica normalizada. Con radio 1, por ejemplo, entra A y no entra B.